# Topic Modeling on DialogSum-RU — SOTA Pipeline

This notebook performs topic discovery on the `topic` column of the [`d0rj/dialogsum-ru`](https://huggingface.co/datasets/d0rj/dialogsum-ru) dataset using a modern, transparent pipeline:

1. **Sentence embeddings** with a multilingual `sentence-transformers` model on GPU (A100).
2. **Dimensionality reduction** with UMAP.
3. **Density-based clustering** with HDBSCAN (handles variable cluster shapes and outliers).
4. **Cluster interpretation**: TF-IDF keywords + representative original topics + optional LLM-generated `cluster_name` / `cluster_description`.
5. **Classical baselines** (LDA, NMF) for comparison.

The output is an annotated table where every DialogSum-RU row carries a stable `cluster_id`, a human-readable `cluster_name`, and a `cluster_description` — usable for downstream tasks (stratification, rare-topic consolidation, label hierarchies, online assignment of new dialogues).

> **Hardware:** designed to run on an A100 GPU (Colab/Kaggle/local). Falls back to CPU if no CUDA device is available.  
> **LLM step:** disabled by default (`RUN_LLM_INTERPRETATION = False`). Heuristic naming from top keywords is used as a fallback.

## Cell 1 — Installs

In [ ]:
# cell 1: installs
# Uncomment when running in a fresh environment (Colab/Kaggle).
# !pip install -q pandas numpy pyarrow matplotlib seaborn scikit-learn \
#     sentence-transformers hdbscan umap-learn huggingface_hub fsspec

## Cell 2 — Imports and settings

In [ ]:
# cell 2: imports and settings
import os
import re
import json
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sentence_transformers import SentenceTransformer
import umap
import hdbscan

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.metrics import silhouette_score

try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
if HAS_TORCH:
    torch.manual_seed(RANDOM_STATE)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_STATE)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid')

DEVICE = 'cuda' if (HAS_TORCH and torch.cuda.is_available()) else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU:    {torch.cuda.get_device_name(0)}')

## Cell 3 — Load and combine DialogSum-RU

In [ ]:
# cell 3: load and combine DialogSum-RU
HF_BASE = 'hf://datasets/d0rj/dialogsum-ru/'
SPLIT_PATHS = {
    'train': 'data/train-00000-of-00001-bcc43b46acda4001.parquet',
    'validation': 'data/validation-00000-of-00001-7e263d81db1c7a12.parquet',
    'test': 'data/test-00000-of-00001-2f13615b955ea947.parquet',
}

frames = []
for split, path in SPLIT_PATHS.items():
    df_part = pd.read_parquet(HF_BASE + path)
    df_part['split'] = split
    print(f'{split:>10}: {df_part.shape}')
    frames.append(df_part)

df_all = pd.concat(frames, ignore_index=True)
print(f'\nCombined df_all shape: {df_all.shape}')
print(f'Columns: {list(df_all.columns)}')
print(f"\nMissing topic values: {df_all['topic'].isna().sum()}")
print('\nSample rows:')
df_all[['id', 'split', 'topic', 'summary']].head(5)

## Cell 4 — Preprocess `topic`

In [ ]:
# cell 4: preprocess topic
_RE_KEEP = re.compile(r'[^a-zA-Zа-яА-ЯёЁ0-9\s]+')
_RE_SPACE = re.compile(r'\s+')

def clean_topic(text):
    """Lowercase, drop chars outside Cyrillic/Latin letters/digits/spaces, collapse whitespace."""
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ''
    s = str(text).lower()
    s = _RE_KEEP.sub(' ', s)
    s = _RE_SPACE.sub(' ', s).strip()
    return s

df_all['topic_clean'] = df_all['topic'].map(clean_topic)

empty_mask = df_all['topic_clean'].str.len() == 0
print(f'Empty topic_clean rows: {int(empty_mask.sum())} / {len(df_all)}')

df_model = df_all[df_all['topic_clean'].str.len() > 0].copy().reset_index(drop=True)
print(f'Modeling set shape: {df_model.shape}')
df_model[['topic', 'topic_clean']].head(10)

## Cell 5 — Build sentence embeddings

Multilingual sentence-transformer (`paraphrase-multilingual-MiniLM-L12-v2`, 384-dim) handles Russian directly. We encode all cleaned topics on GPU and L2-normalize the output — cosine similarity in this space becomes Euclidean distance, which is what HDBSCAN needs after UMAP.

In [ ]:
# cell 5: build sentence embeddings
EMBEDDING_MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
BATCH_SIZE = 64  # comfortable on A100; lower if you hit OOM

embedder = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)
print(f'Loaded {EMBEDDING_MODEL_NAME} on {DEVICE}')

texts = df_model['topic_clean'].tolist()
embeddings = embedder.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f'Embeddings shape: {embeddings.shape}, dtype: {embeddings.dtype}')

## Cell 6 — UMAP + HDBSCAN clustering

UMAP reduces 384-dim embeddings to 5 dims to make density-based clustering tractable and stable; cosine metric matches the embedding geometry. HDBSCAN then labels variable-shape clusters and marks ambiguous points as `-1` (outliers) instead of forcing every document into a cluster.

In [ ]:
# cell 6: cluster embeddings with UMAP + HDBSCAN
UMAP_N_COMPONENTS = 5
UMAP_N_NEIGHBORS = 15
MIN_CLUSTER_SIZE = 30
MIN_SAMPLES = 10

reducer = umap.UMAP(
    n_neighbors=UMAP_N_NEIGHBORS,
    n_components=UMAP_N_COMPONENTS,
    min_dist=0.0,
    metric='cosine',
    random_state=RANDOM_STATE,
)
embeddings_umap = reducer.fit_transform(embeddings)
print(f'UMAP-reduced shape: {embeddings_umap.shape}')

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=MIN_SAMPLES,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True,
)
cluster_labels = clusterer.fit_predict(embeddings_umap)
df_model['cluster_id'] = cluster_labels

if getattr(clusterer, 'probabilities_', None) is not None:
    df_model['cluster_probability'] = clusterer.probabilities_
if getattr(clusterer, 'outlier_scores_', None) is not None:
    df_model['outlier_score'] = clusterer.outlier_scores_

n_clusters = int(df_model.loc[df_model['cluster_id'] != -1, 'cluster_id'].nunique())
n_outliers = int((df_model['cluster_id'] == -1).sum())
print(f'Number of clusters (excluding -1): {n_clusters}')
print(f'Outliers (-1): {n_outliers} ({n_outliers / len(df_model):.1%})')

## Cell 7 — Inspect cluster sizes and quality

In [ ]:
# cell 7: inspect clusters and quality metrics
size_counts = df_model['cluster_id'].value_counts().sort_values(ascending=False)
print('Top 25 cluster sizes (cluster_id : count):')
print(size_counts.head(25))

non_outlier_sizes = size_counts[size_counts.index != -1]
top_show = non_outlier_sizes.head(30)
if len(top_show) > 0:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(top_show.index.astype(str), top_show.values, color='steelblue')
    ax.set_title('Top 30 cluster sizes (HDBSCAN, excluding outliers)')
    ax.set_xlabel('cluster_id')
    ax.set_ylabel('# topics')
    plt.xticks(rotation=60)
    plt.tight_layout()
    plt.show()

# Silhouette score on the UMAP space (same space HDBSCAN saw), excluding -1.
# Sample-cap to keep runtime reasonable on large datasets.
sil = None
valid_mask = df_model['cluster_id'] != -1
if valid_mask.sum() >= 2 and df_model.loc[valid_mask, 'cluster_id'].nunique() >= 2:
    sample_idx = np.where(valid_mask.values)[0]
    if len(sample_idx) > 20000:
        sample_idx = np.random.RandomState(RANDOM_STATE).choice(sample_idx, 20000, replace=False)
    sil = float(silhouette_score(
        embeddings_umap[sample_idx],
        df_model['cluster_id'].values[sample_idx],
        metric='euclidean',
    ))
    print(f'Silhouette score (UMAP space, excl. -1): {sil:.4f}')
else:
    print('Silhouette score: not computable (need >= 2 clusters with >= 2 docs each).')

## Cell 8 — Cluster keywords and representative examples

We compute per-cluster TF-IDF over concatenated topics and surface the top n-grams. Representative topics are sampled by HDBSCAN membership probability when available; otherwise we take the first rows in the cluster.

In [ ]:
# cell 8: extract keywords and representative examples
TOP_K_KEYWORDS = 10
TOP_K_EXAMPLES = 5

# Build one pseudo-document per cluster (incl. -1 for visibility)
cluster_docs = (
    df_model.groupby('cluster_id')['topic_clean']
    .apply(lambda s: ' '.join(s.tolist()))
    .sort_index()
)
cluster_ids_sorted = cluster_docs.index.tolist()

kw_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=5000,
)
tfidf_matrix = kw_vectorizer.fit_transform(cluster_docs.values)
kw_vocab = np.array(kw_vectorizer.get_feature_names_out())

def top_keywords_for_row(row, k=TOP_K_KEYWORDS):
    arr = row.toarray().ravel()
    if arr.sum() == 0:
        return []
    idx = np.argsort(-arr)[:k]
    return [kw_vocab[i] for i in idx if arr[i] > 0]

rows = []
for pos, cid in enumerate(cluster_ids_sorted):
    sub = df_model[df_model['cluster_id'] == cid]
    keywords = top_keywords_for_row(tfidf_matrix[pos])
    if 'cluster_probability' in sub.columns:
        sub_sorted = sub.sort_values('cluster_probability', ascending=False)
    else:
        sub_sorted = sub
    examples = sub_sorted['topic'].head(TOP_K_EXAMPLES).tolist()
    rows.append({
        'cluster_id': int(cid),
        'cluster_size': int(len(sub)),
        'top_keywords': keywords,
        'representative_topics': examples,
    })

topics_df = pd.DataFrame(rows).sort_values('cluster_size', ascending=False).reset_index(drop=True)
print(f'Discovered {len(topics_df)} cluster rows (including outliers row if present).')
topics_df.head(15)

## Cell 9 — LLM interpretation (optional)

We prepare a Russian-language prompt per cluster that asks an LLM to produce a short theme name (2–5 words) and a 1–2 sentence description, grounded in keywords + representative original topics. The actual API call is gated by `RUN_LLM_INTERPRETATION`; when disabled (default), we fall back to a deterministic heuristic so the notebook is fully runnable offline.

In [ ]:
# cell 9: LLM interpretation prompts and optional execution placeholder
RUN_LLM_INTERPRETATION = False  # set True only if you wire up an API client below

def build_cluster_prompt(cluster_id, keywords, representative_topics):
    kw_str = ', '.join(keywords) if keywords else '(нет ключевых слов)'
    ex_str = '\n'.join(f'- {t}' for t in representative_topics) if representative_topics else '(нет примеров)'
    return (
        'Ты — аналитик диалогов. Ниже приведены ключевые слова и примеры тем '
        f'кластера #{cluster_id} из русскоязычного датасета диалогов DialogSum-RU.\n\n'
        f'Ключевые слова: {kw_str}\n\n'
        f'Примеры исходных тем:\n{ex_str}\n\n'
        'Задача: предложи (1) короткое название темы на русском языке '
        '(2–5 слов) и (2) описание из 1–2 предложений, объясняющее, '
        'о чём этот кластер. Верни строго JSON вида '
        '{"cluster_name": "...", "cluster_description": "..."}.'
    )

def heuristic_name_and_description(keywords):
    if not keywords:
        return ('Без ключевых слов',
                'Автоматическое описание не сгенерировано; см. keywords/examples.')
    name = ' / '.join(keywords[:3])
    return (name,
            'Автоматическое описание не сгенерировано; см. keywords/examples.')

def call_llm_placeholder(prompt):
    """Placeholder for an LLM call. Replace with your provider of choice, e.g.:

        from openai import OpenAI
        client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
        resp = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': prompt}],
            response_format={'type': 'json_object'},
        )
        return json.loads(resp.choices[0].message.content)

    Equivalent options: Perplexity API, Anthropic Claude, local vLLM, etc.
    """
    raise NotImplementedError(
        'Wire up an LLM client before setting RUN_LLM_INTERPRETATION=True.'
    )

cluster_names = {}
cluster_descriptions = {}

for _, r in topics_df.iterrows():
    cid = r['cluster_id']
    kws = r['top_keywords']
    if cid == -1:
        cluster_names[cid] = 'Выбросы / разное'
        cluster_descriptions[cid] = (
            'Темы, не отнесённые HDBSCAN ни к одному плотному кластеру.'
        )
        continue
    if RUN_LLM_INTERPRETATION:
        try:
            prompt = build_cluster_prompt(cid, kws, r['representative_topics'])
            parsed = call_llm_placeholder(prompt)
            name = (parsed.get('cluster_name') or '').strip()
            desc = (parsed.get('cluster_description') or '').strip()
            if not name or not desc:
                hname, hdesc = heuristic_name_and_description(kws)
                name = name or hname
                desc = desc or hdesc
            cluster_names[cid] = name
            cluster_descriptions[cid] = desc
        except Exception as e:
            print(f'LLM call failed for cluster {cid}: {e}; using heuristic fallback.')
            name, desc = heuristic_name_and_description(kws)
            cluster_names[cid] = name
            cluster_descriptions[cid] = desc
    else:
        name, desc = heuristic_name_and_description(kws)
        cluster_names[cid] = name
        cluster_descriptions[cid] = desc

topics_df['cluster_name'] = topics_df['cluster_id'].map(cluster_names)
topics_df['cluster_description'] = topics_df['cluster_id'].map(cluster_descriptions)

df_model['cluster_name'] = df_model['cluster_id'].map(cluster_names)
df_model['cluster_description'] = df_model['cluster_id'].map(cluster_descriptions)

topics_df[['cluster_id', 'cluster_size', 'cluster_name', 'top_keywords']].head(15)

## Cell 10 — Lightweight LDA / NMF baseline

Classical bag-of-words topic models on `topic_clean` for `k ∈ {20, 40}`. This is a sanity baseline — not a full sweep — to demonstrate the qualitative gap with the embedding pipeline.

In [ ]:
# cell 10: lightweight LDA/NMF baseline
K_VALUES = [20, 40]
N_TOP_WORDS = 8

count_vec = CountVectorizer(min_df=2, max_df=0.95, max_features=10000, ngram_range=(1, 1))
X_counts = count_vec.fit_transform(df_model['topic_clean'])
count_vocab = np.array(count_vec.get_feature_names_out())

tfidf_vec = TfidfVectorizer(min_df=2, max_df=0.95, max_features=10000, ngram_range=(1, 1))
X_tfidf = tfidf_vec.fit_transform(df_model['topic_clean'])
tfidf_vocab = np.array(tfidf_vec.get_feature_names_out())

def print_top_words(model, vocab, label, k=N_TOP_WORDS):
    for i, comp in enumerate(model.components_):
        top = vocab[np.argsort(-comp)[:k]]
        print(f'  {label} topic {i:>2}: {", ".join(top)}')

baseline_stats = []
for k in K_VALUES:
    print(f'\n=== LDA, k={k} ===')
    lda = LatentDirichletAllocation(
        n_components=k,
        random_state=RANDOM_STATE,
        learning_method='online',
        max_iter=10,
    )
    lda_doc_topic = lda.fit_transform(X_counts)
    lda_assign = lda_doc_topic.argmax(axis=1)
    print_top_words(lda, count_vocab, 'LDA')
    sizes = pd.Series(lda_assign).value_counts().sort_values(ascending=False)
    print(f'  Cluster-size distribution (top 5): {sizes.head(5).to_dict()}')
    baseline_stats.append(('LDA', k, sizes))

    print(f'\n=== NMF, k={k} ===')
    nmf = NMF(
        n_components=k,
        random_state=RANDOM_STATE,
        init='nndsvda',
        max_iter=300,
    )
    nmf_doc_topic = nmf.fit_transform(X_tfidf)
    nmf_assign = nmf_doc_topic.argmax(axis=1)
    print_top_words(nmf, tfidf_vocab, 'NMF')
    sizes = pd.Series(nmf_assign).value_counts().sort_values(ascending=False)
    print(f'  Cluster-size distribution (top 5): {sizes.head(5).to_dict()}')
    baseline_stats.append(('NMF', k, sizes))

## Cell 11 — Compare approaches

In [ ]:
# cell 11: compare approaches
comparison_rows = []

emb_outliers = int((df_model['cluster_id'] == -1).sum())
emb_n_clusters = int(df_model.loc[df_model['cluster_id'] != -1, 'cluster_id'].nunique())
comparison_rows.append({
    'method': 'Embeddings + UMAP + HDBSCAN',
    'k_or_clusters': emb_n_clusters,
    'outliers': emb_outliers,
    'silhouette': sil,
    'notes': 'Multilingual semantic space; outlier-aware; LLM-friendly cluster labels.',
})

for method, k, _sizes in baseline_stats:
    comparison_rows.append({
        'method': method,
        'k_or_clusters': k,
        'outliers': 0,
        'silhouette': None,
        'notes': 'Bag-of-words; fixed k; every doc assigned; sensitive to synonyms/morphology.',
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

**Why embeddings + UMAP + HDBSCAN + LLM is the SOTA approach here**

- **Synonymy & paraphrase robustness.** Multilingual sentence embeddings put `«заказ столика»` and `«бронирование ресторана»` near each other; LDA/NMF treat them as disjoint bag-of-words tokens unless surface forms overlap.
- **Morphology.** Russian inflection fragments BOW counts across cases/numbers/genders; transformer embeddings are agnostic to surface form.
- **Cluster shape & count.** HDBSCAN infers the number of clusters from density and allows variable-size, non-spherical clusters. LDA/NMF require a fixed `k` and assign every document — including noise — to some topic.
- **Outlier handling.** HDBSCAN's `-1` label is a feature, not a bug: it isolates ambiguous or rare topics rather than diluting real clusters.
- **Interpretability.** TF-IDF keywords + LLM-generated `cluster_name`/`cluster_description` produce labels a human can read at a glance; LDA/NMF top-word lists usually still need manual naming.
- **Reusability.** Embeddings are normalized and HDBSCAN was fit with `prediction_data=True`, so new dialogues can be assigned to existing clusters via `hdbscan.approximate_predict` or nearest-centroid lookup in UMAP space.

## Cell 12 — Save final topic clustering results

In [ ]:
# cell 12: save final topic clustering results
DRIVE_DIR = '/content/drive/MyDrive/russian-dialogue-intent-thesis/results/tables'
LOCAL_DIR_PRIMARY = '/home/user/workspace/results/tables'
LOCAL_DIR_FALLBACK = 'results/tables'

def pick_output_dir():
    if os.path.isdir(DRIVE_DIR):
        return DRIVE_DIR
    for candidate in (LOCAL_DIR_PRIMARY, LOCAL_DIR_FALLBACK):
        try:
            os.makedirs(candidate, exist_ok=True)
            return candidate
        except OSError:
            continue
    return '.'

out_dir = pick_output_dir()
print(f'Output directory: {out_dir}')

# Build the final annotated frame on df_all so every original row is preserved.
join_cols = ['cluster_id', 'cluster_name', 'cluster_description']
for opt in ('cluster_probability', 'outlier_score'):
    if opt in df_model.columns:
        join_cols.append(opt)

df_final = df_all.merge(
    df_model[['id', 'split'] + join_cols],
    on=['id', 'split'],
    how='left',
)

base_cols = ['id', 'split', 'dialogue', 'summary', 'topic', 'topic_clean',
             'cluster_id', 'cluster_name', 'cluster_description']
extra_cols = [c for c in ('cluster_probability', 'outlier_score') if c in df_final.columns]
df_final = df_final[base_cols + extra_cols]

csv_path = os.path.join(out_dir, 'dialogsum_ru_topic_clusters_sota.csv')
parquet_path = os.path.join(out_dir, 'dialogsum_ru_topic_clusters_sota.parquet')
df_final.to_csv(csv_path, index=False)
df_final.to_parquet(parquet_path, index=False)

print(f'Wrote CSV:     {csv_path}')
print(f'Wrote Parquet: {parquet_path}')
print(f'Final columns: {list(df_final.columns)}')
print(f'Final shape:   {df_final.shape}')

## Cell 13 — Final conclusions

The annotated table (`dialogsum_ru_topic_clusters_sota.csv` / `.parquet`) gives every DialogSum-RU row a semantic `cluster_id` plus a human-readable `cluster_name` and `cluster_description`. Refer to the cell-7 plot, the silhouette printout, and the `topics_df` from cell 8 for the actual cluster inventory — concrete numbers depend on the run and are not hard-coded here.

**Suggested downstream uses**

- **Grouping similar topics.** Treat `cluster_id` (or `cluster_name`) as a coarse label that collapses surface variants of the same theme (e.g. paraphrased orderings of the same intent).
- **Rare-topic consolidation.** Small clusters and the `-1` outlier bucket can be merged into an "other / rare" bucket for downstream classifiers, or used to flag candidates for manual review.
- **Stratification.** Use `cluster_id` for stratified train/val/test splitting or for balanced sampling when building annotation batches and few-shot demonstrations.
- **Label hierarchy.** Cluster names can serve as the top tier of an intent/topic taxonomy, with finer-grained intents annotated underneath.
- **Online assignment.** New dialogues can be embedded with the same model and assigned to existing clusters via `hdbscan.approximate_predict` (since `prediction_data=True`) or nearest-centroid lookup in UMAP space.

To upgrade quality further: enable `RUN_LLM_INTERPRETATION = True` after wiring up an LLM client in cell 9, and/or sweep `min_cluster_size` and `n_neighbors` while keeping the rest of the pipeline fixed.